# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 03 - Leakage-safe GNN playlist completion

This notebook trains and evaluates a recommendation model from the integrated playlist–track graph produced by notebooks 01, 01b, 02, 02b, and 02c. It also exports a `.pt` inference bundle and a compressed track catalog for a later web demonstration.

### Model decision

The task is **supervised implicit-feedback link prediction**. Existing playlist–track edges are positive labels; sampled unobserved pairs are training negatives. It is not fully unsupervised because held-out edges provide validation and test targets, but it does not require explicit ratings or dislikes. Pairwise Bayesian Personalized Ranking (BPR) optimizes each observed track above an unseen track.

The collaborative core is [LightGCN](https://arxiv.org/abs/2002.02126): two linear neighborhood-aggregation layers over the playlist–track bipartite graph. The ranking objective follows [BPR](https://arxiv.org/abs/1205.2618). PyTorch sparse matrix multiplication is used for graph propagation, and the saved artifact follows PyTorch's recommended [state-dict approach](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

## Modeling and evaluation principles

1. **Split links before graph construction.** Validation and test edges never enter message passing, degree normalization, negative labels, early stopping, or hyperparameter selection.
2. **Split within each eligible playlist.** Approximately 70% of each playlist's tracks are training seeds, 15% validation, and 15% test. Playlists with fewer than five tracks are training-only because a three-way split would not leave a useful seed set.
3. **Use validation once per training decision; keep test sealed.** Epoch and content-blend weight are selected on validation. Test metrics are calculated only after the checkpoint is fixed.
4. **Avoid false negatives.** Negative sampling excludes every known positive edge, including hidden validation/test positives, without revealing them as positive inputs to the model.
5. **Use ranking metrics.** Recall@K, NDCG@K, HitRate@K, MRR@K, catalog coverage, and tail-item share are reported against identical sampled candidate sets for all models.
6. **Control popularity and metadata bias.** Popularity is a baseline, not a safe GNN input. Audio information is an optional validation-selected reranking signal because 02c showed that its coverage is sparse and non-random.
7. **Do not call the website score a probability.** The displayed match percentage is a validation-calibrated relative fit score on a 0–100 scale. It is calibrated from balanced hidden-positive and unseen validation examples, not online user likes.

## 1. Environment, GPU selection, and reproducibility

In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import random
import sys
import time
from pathlib import Path


def find_project_root():
    required = Path("data/processed/gnn_integrated/playlist_track_edges.parquet")
    for anchor in [Path.cwd().resolve(), Path(sys.executable).resolve()]:
        for parent in [anchor, *anchor.parents]:
            for candidate in [parent, parent / "art_xharra"]:
                if (candidate / required).exists():
                    return candidate.resolve()
    raise FileNotFoundError("Integrated graph outputs are missing. Run notebook 02 first.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
JDK_ROOT = PROJECT_ROOT / ".tools/jdk17/jdk-17.0.20+8"
HADOOP_ROOT = PROJECT_ROOT / ".tools/hadoop"
os.environ["JAVA_HOME"] = str(JDK_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = str(JDK_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")
if os.name == "nt":
    os.environ["HADOOP_HOME"] = str(HADOOP_ROOT)
    os.environ["PATH"] = str(HADOOP_ROOT / "bin") + os.pathsep + os.environ["PATH"]

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
import torch.nn.functional as torch_f

SEED = 20260828
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    device_description = f"{gpu.name} ({gpu.total_memory / 2**30:.1f} GiB, CUDA {torch.version.cuda})"
else:
    device_description = "CPU fallback"

spark = (
    SparkSession.builder.master("local[*]")
    .appName("Leakage-Safe-LightGCN")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.driver.memory", "5g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

GRAPH_ROOT = PROJECT_ROOT / "data/processed/gnn_integrated"
SPLIT_ROOT = PROJECT_ROOT / "data/processed/modeling/playlist_track_split.parquet"
MODEL_ROOT = PROJECT_ROOT / "models"
REPORT_ROOT = PROJECT_ROOT / "reports"
FIGURE_ROOT = REPORT_ROOT / "figures/model"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180, "axes.titleweight": "bold"})
print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"Training     : {device_description}")
print(f"Split seed   : {SEED}")

**Observed.** The notebook automatically uses the CUDA GPU when available and otherwise remains executable on CPU. Random seeds and deterministic split hashes make the data partition and candidate sets reproducible.

## 2. Load the validated graph

In [ ]:
playlist_nodes = spark.read.parquet(str(GRAPH_ROOT / "playlist_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
track_nodes = spark.read.parquet(str(GRAPH_ROOT / "track_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
playlist_track_edges = spark.read.parquet(str(GRAPH_ROOT / "playlist_track_edges.parquet")).persist(StorageLevel.MEMORY_AND_DISK)

graph_counts = {
    "playlists": playlist_nodes.count(),
    "tracks": track_nodes.count(),
    "playlist_track_edges": playlist_track_edges.count(),
    "eligible_playlists": playlist_nodes.where("eligible_for_link_prediction = 1").count(),
}
assert playlist_nodes.agg(F.min("playlist_node_id"), F.max("playlist_node_id")).first() == (0, graph_counts["playlists"] - 1)
assert track_nodes.agg(F.min("track_node_id"), F.max("track_node_id")).first() == (0, graph_counts["tracks"] - 1)
display(pd.DataFrame(graph_counts.items(), columns=["graph_object", "count"]))

**Observed.** The contiguous node IDs can be used directly as embedding indices. Only the 18,409 playlists with at least five tracks contribute validation and test labels; all 18,829 connected playlists can contribute training signal.

## 3. Deterministic 70% / 15% / 15% per-playlist edge split

In [ ]:
edge_window = Window.partitionBy("src_playlist_node_id")
rank_window = edge_window.orderBy(
    F.xxhash64("src_playlist_node_id", "dst_track_node_id", F.lit(SEED)),
    "dst_track_node_id",
)

ranked_edges = (
    playlist_track_edges.select(
        "src_playlist_node_id", "dst_track_node_id", "playlist_id", "spotify_track_id", "spud_track_id"
    )
    .join(
        playlist_nodes.select("playlist_node_id", "eligible_for_link_prediction", "playlist_size_band"),
        F.col("src_playlist_node_id") == F.col("playlist_node_id"),
        "inner",
    )
    .drop("playlist_node_id")
    .withColumn("edge_count_in_playlist", F.count("*").over(edge_window))
    .withColumn("edge_rank", F.row_number().over(rank_window))
    .withColumn(
        "validation_count",
        F.when(
            F.col("eligible_for_link_prediction") == 1,
            F.greatest(F.lit(1), F.round(0.15 * F.col("edge_count_in_playlist")).cast("int")),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "test_count",
        F.when(
            F.col("eligible_for_link_prediction") == 1,
            F.greatest(F.lit(1), F.round(0.15 * F.col("edge_count_in_playlist")).cast("int")),
        ).otherwise(F.lit(0)),
    )
    .withColumn("training_count", F.col("edge_count_in_playlist") - F.col("validation_count") - F.col("test_count"))
    .withColumn(
        "split",
        F.when(F.col("edge_rank") <= F.col("training_count"), "train")
        .when(F.col("edge_rank") <= F.col("training_count") + F.col("validation_count"), "validation")
        .otherwise("test"),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

split_counts_pdf = (
    ranked_edges.groupBy("split").count()
    .withColumn("percentage", 100 * F.col("count") / F.lit(graph_counts["playlist_track_edges"]))
    .orderBy(F.when(F.col("split") == "train", 0).when(F.col("split") == "validation", 1).otherwise(2))
    .toPandas()
)

split_pairs = {
    name: ranked_edges.where(F.col("split") == name).select("src_playlist_node_id", "dst_track_node_id")
    for name in ["train", "validation", "test"]
}
overlap_counts = {
    "train_validation": split_pairs["train"].join(split_pairs["validation"], ["src_playlist_node_id", "dst_track_node_id"]).count(),
    "train_test": split_pairs["train"].join(split_pairs["test"], ["src_playlist_node_id", "dst_track_node_id"]).count(),
    "validation_test": split_pairs["validation"].join(split_pairs["test"], ["src_playlist_node_id", "dst_track_node_id"]).count(),
}
eligible_split_audit = (
    ranked_edges.where("eligible_for_link_prediction = 1")
    .groupBy("src_playlist_node_id")
    .pivot("split", ["train", "validation", "test"]).count().na.fill(0)
)
assert sum(split_counts_pdf["count"]) == graph_counts["playlist_track_edges"]
assert all(value == 0 for value in overlap_counts.values())
assert eligible_split_audit.where("train < 3 OR validation < 1 OR test < 1").count() == 0

(
    ranked_edges.select(
        "src_playlist_node_id", "dst_track_node_id", "playlist_id", "spotify_track_id",
        "spud_track_id", "playlist_size_band", "eligible_for_link_prediction", "split",
    )
    .write.mode("overwrite").option("compression", "snappy").parquet(str(SPLIT_ROOT))
)
split_counts_pdf.to_csv(REPORT_ROOT / "model_edge_split_counts.csv", index=False)
display(split_counts_pdf.round(3))
print("Overlap audit:", overlap_counts)
print(f"Saved split table: {SPLIT_ROOT.relative_to(PROJECT_ROOT)}")

**Observed.** The proportions are close to 70/15/15 while preserving at least three visible training tracks and one hidden edge in each evaluation partition for every eligible playlist. The overlap audit must remain zero. Non-eligible short playlists are training-only.

## 4. Materialize arrays, safe song features, and candidate protocol

In [ ]:
def edge_arrays(split_name):
    pdf = (
        ranked_edges.where(F.col("split") == split_name)
        .select("src_playlist_node_id", "dst_track_node_id")
        .orderBy("src_playlist_node_id", "dst_track_node_id").toPandas()
    )
    return (
        pdf["src_playlist_node_id"].to_numpy(dtype=np.int64),
        pdf["dst_track_node_id"].to_numpy(dtype=np.int64),
    )


train_users, train_items = edge_arrays("train")
validation_users, validation_items = edge_arrays("validation")
test_users, test_items = edge_arrays("test")
num_playlists = graph_counts["playlists"]
num_tracks = graph_counts["tracks"]

track_catalog_pdf = (
    track_nodes.select(
        "track_node_id", "spotify_track_id", "track_title", "artist_name",
        "spotify_artist_id", "spud_popularity", "audio_features_available", "features_safe",
    ).orderBy("track_node_id").toPandas()
)
assert np.array_equal(track_catalog_pdf["track_node_id"].to_numpy(), np.arange(num_tracks))
safe_features = np.stack(track_catalog_pdf.pop("features_safe").to_numpy()).astype(np.float32)

# Indices 2:33 are leakage-safe catalog audio/key/time-signature features. Missing-audio rows are zeroed.
audio_available = track_catalog_pdf["audio_features_available"].to_numpy(dtype=np.int8).astype(bool)
audio_features = safe_features[:, 2:33].copy()
audio_features[~audio_available] = 0.0
audio_norms = np.linalg.norm(audio_features, axis=1, keepdims=True)
audio_features = audio_features / np.maximum(audio_norms, 1.0e-12)
del safe_features, audio_norms

playlist_catalog_pdf = (
    playlist_nodes.select("playlist_node_id", "playlist_id", "playlist_title", "playlist_size_band")
    .orderBy("playlist_node_id").toPandas()
)
track_catalog_path = MODEL_ROOT / "lightgcn_track_catalog.csv.gz"
track_catalog_pdf.to_csv(track_catalog_path, index=False, compression="gzip")

all_users = np.concatenate([train_users, validation_users, test_users])
all_items = np.concatenate([train_items, validation_items, test_items])
known_positive_codes = np.sort(all_users * np.int64(num_tracks) + all_items)


def sample_unseen_items(users, rng):
    users = np.asarray(users, dtype=np.int64)
    negatives = rng.integers(0, num_tracks, size=len(users), dtype=np.int64)
    while True:
        codes = users * np.int64(num_tracks) + negatives
        positions = np.searchsorted(known_positive_codes, codes)
        collisions = (positions < len(known_positive_codes)) & (known_positive_codes[np.minimum(positions, len(known_positive_codes) - 1)] == codes)
        if not collisions.any():
            return negatives
        negatives[collisions] = rng.integers(0, num_tracks, size=int(collisions.sum()), dtype=np.int64)


def grouped_items(users, items):
    grouped = [np.empty(0, dtype=np.int64) for _ in range(num_playlists)]
    if len(users) == 0:
        return grouped
    order = np.argsort(users, kind="stable")
    sorted_users, sorted_items = users[order], items[order]
    unique_users, starts = np.unique(sorted_users, return_index=True)
    ends = np.r_[starts[1:], len(sorted_users)]
    for user, start, end in zip(unique_users, starts, ends):
        grouped[int(user)] = sorted_items[start:end]
    return grouped


train_items_by_playlist = grouped_items(train_users, train_items)
validation_items_by_playlist = grouped_items(validation_users, validation_items)
test_items_by_playlist = grouped_items(test_users, test_items)


def sample_unique_unseen(user, count, seed_offset):
    rng = np.random.default_rng(SEED + seed_offset + int(user) * 104729)
    selected = np.empty(0, dtype=np.int64)
    while len(selected) < count:
        draw = rng.integers(0, num_tracks, size=max(64, 2 * (count - len(selected))), dtype=np.int64)
        codes = np.int64(user) * np.int64(num_tracks) + draw
        positions = np.searchsorted(known_positive_codes, codes)
        known = (positions < len(known_positive_codes)) & (known_positive_codes[np.minimum(positions, len(known_positive_codes) - 1)] == codes)
        selected = np.unique(np.concatenate([selected, draw[~known]]))
    return selected[:count]


def build_candidate_sets(hidden_by_playlist, negatives_per_playlist, seed_offset):
    candidates = {}
    for user, positives in enumerate(hidden_by_playlist):
        if len(positives):
            negatives = sample_unique_unseen(user, negatives_per_playlist, seed_offset)
            candidates[user] = (np.concatenate([positives, negatives]), positives)
    return candidates


VALIDATION_NEGATIVES = 200
TEST_NEGATIVES = 200
validation_candidates = build_candidate_sets(validation_items_by_playlist, VALIDATION_NEGATIVES, 10_000)
test_candidates = build_candidate_sets(test_items_by_playlist, TEST_NEGATIVES, 20_000)
print(f"Train/validation/test edges: {len(train_items):,} / {len(validation_items):,} / {len(test_items):,}")
print(f"Validation/test playlists   : {len(validation_candidates):,} / {len(test_candidates):,}")
print(f"Candidate protocol          : all hidden positives + {VALIDATION_NEGATIVES} reproducible unseen negatives")
print(f"Track catalog               : {track_catalog_path.relative_to(PROJECT_ROOT)}")

**Observed.** Candidate negatives exclude every observed or held-out positive. All models receive identical candidates, so comparisons are fair. Metrics are sampled-ranking estimates against 200 unseen negatives per playlist; they should not be compared directly with full-catalog metrics from another study.

## 5. LightGCN graph and model definition

In [ ]:
def normalized_training_adjacency(users, items, device):
    user_nodes = torch.from_numpy(users).long()
    item_nodes = torch.from_numpy(items).long() + num_playlists
    src = torch.cat([user_nodes, item_nodes])
    dst = torch.cat([item_nodes, user_nodes])
    node_count = num_playlists + num_tracks
    degree = torch.bincount(src, minlength=node_count).float()
    values = torch.rsqrt(degree[src].clamp_min(1.0) * degree[dst].clamp_min(1.0))
    adjacency = torch.sparse_coo_tensor(
        torch.stack([src, dst]), values, (node_count, node_count), dtype=torch.float32,
        check_invariants=False,
    ).coalesce()
    return adjacency.to(device)


class LightGCN(nn.Module):
    def __init__(self, playlist_count, track_count, embedding_dim=32, layers=2):
        super().__init__()
        self.playlist_count = playlist_count
        self.track_count = track_count
        self.layers = layers
        self.embedding = nn.Embedding(playlist_count + track_count, embedding_dim)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self, adjacency):
        current = self.embedding.weight
        propagated = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(adjacency, current)
            propagated.append(current)
        final = torch.stack(propagated, dim=0).mean(dim=0)
        return final[: self.playlist_count], final[self.playlist_count :]


EMBEDDING_DIM = 32
LAYERS = 2
adjacency = normalized_training_adjacency(train_users, train_items, DEVICE)
model = LightGCN(num_playlists, num_tracks, EMBEDDING_DIM, LAYERS).to(DEVICE)

train_degree = np.bincount(train_items, minlength=num_tracks).astype(np.float32)
zero_degree_tracks = int((train_degree == 0).sum())
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f"Train-only directed adjacency entries: {adjacency._nnz():,}")
print(f"Trainable parameters                : {parameter_count:,}")
print(f"Tracks absent from training graph   : {zero_degree_tracks:,}")
if DEVICE.type == "cuda":
    print(f"GPU memory after setup              : {torch.cuda.memory_allocated() / 2**20:.1f} MiB")

**Observed.** The adjacency contains training links only; validation and test tracks cannot influence propagation. LightGCN deliberately uses no nonlinear activation or feature transformation in its collaborative layers. Tracks that appear only in hidden partitions cannot learn collaborative neighborhoods, which is a genuine cold-start limitation rather than something to patch with leakage.

## 6. Ranking metrics and validation-only early stopping

In [ ]:
def unit_rows(values):
    values = np.asarray(values, dtype=np.float32)
    return values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1.0e-12)


def ranking_metrics(candidate_sets, score_function, ks=(10, 20), limit_users=None):
    users = np.array(sorted(candidate_sets), dtype=np.int64)
    if limit_users is not None and len(users) > limit_users:
        rng = np.random.default_rng(SEED + 777)
        users = np.sort(rng.choice(users, size=limit_users, replace=False))
    accumulators = {k: {"recall": [], "ndcg": [], "hitrate": [], "mrr": []} for k in ks}
    recommended = {k: set() for k in ks}
    tail_hits = {k: [] for k in ks}
    for user in users:
        candidates, positives = candidate_sets[int(user)]
        scores = np.asarray(score_function(int(user), candidates), dtype=np.float64)
        order = np.lexsort((candidates, -scores))
        ranked = candidates[order]
        positive_set = set(int(value) for value in positives)
        for k in ks:
            top = ranked[:k]
            hits = np.array([int(item) in positive_set for item in top], dtype=bool)
            hit_positions = np.flatnonzero(hits)
            recall = hits.sum() / len(positive_set)
            dcg = np.sum(hits / np.log2(np.arange(2, len(hits) + 2)))
            ideal_length = min(k, len(positive_set))
            idcg = np.sum(1 / np.log2(np.arange(2, ideal_length + 2)))
            accumulators[k]["recall"].append(float(recall))
            accumulators[k]["ndcg"].append(float(dcg / idcg))
            accumulators[k]["hitrate"].append(float(hits.any()))
            accumulators[k]["mrr"].append(float(1 / (hit_positions[0] + 1)) if len(hit_positions) else 0.0)
            recommended[k].update(int(item) for item in top)
            tail_hits[k].extend((train_degree[top] <= 4).astype(float).tolist())
    rows = []
    for k in ks:
        rows.append({
            "k": k,
            "evaluated_playlists": len(users),
            "recall": np.mean(accumulators[k]["recall"]),
            "ndcg": np.mean(accumulators[k]["ndcg"]),
            "hitrate": np.mean(accumulators[k]["hitrate"]),
            "mrr": np.mean(accumulators[k]["mrr"]),
            "catalog_coverage": len(recommended[k]) / num_tracks,
            "tail_recommendation_share": np.mean(tail_hits[k]),
        })
    return pd.DataFrame(rows)


TUNING_PLAYLISTS = 2_500
EPOCHS = 160
PATIENCE = 8
MIN_VALIDATION_IMPROVEMENT = 1.0e-4
SAMPLES_PER_EPOCH = min(300_000, len(train_items))
LEARNING_RATE = 0.01
L2_REGULARIZATION = 1.0e-5
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
training_rng = np.random.default_rng(SEED + 1)
history = []
best_validation_ndcg = -np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0


def quick_validation_ndcg(playlist_embeddings, track_embeddings):
    playlist_embeddings = unit_rows(playlist_embeddings)
    track_embeddings = unit_rows(track_embeddings)
    def scorer(user, candidates):
        return track_embeddings[candidates] @ playlist_embeddings[user]
    return float(
        ranking_metrics(
            validation_candidates, scorer, ks=(10,), limit_users=TUNING_PLAYLISTS
        ).loc[0, "ndcg"]
    )


for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    sampled_indices = training_rng.integers(0, len(train_items), size=SAMPLES_PER_EPOCH)
    batch_users_np = train_users[sampled_indices]
    positive_items_np = train_items[sampled_indices]
    negative_items_np = sample_unseen_items(batch_users_np, training_rng)
    batch_users = torch.from_numpy(batch_users_np).long().to(DEVICE)
    positive_items = torch.from_numpy(positive_items_np).long().to(DEVICE)
    negative_items = torch.from_numpy(negative_items_np).long().to(DEVICE)

    playlist_embeddings, track_embeddings = model(adjacency)
    positive_scores = (playlist_embeddings[batch_users] * track_embeddings[positive_items]).sum(dim=1)
    negative_scores = (playlist_embeddings[batch_users] * track_embeddings[negative_items]).sum(dim=1)
    ranking_loss = -torch_f.logsigmoid(positive_scores - negative_scores).mean()
    raw = model.embedding.weight
    raw_users = raw[batch_users]
    raw_positive = raw[num_playlists + positive_items]
    raw_negative = raw[num_playlists + negative_items]
    regularization = L2_REGULARIZATION * (
        raw_users.square().sum(dim=1) + raw_positive.square().sum(dim=1) + raw_negative.square().sum(dim=1)
    ).mean()
    loss = ranking_loss + regularization
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        validation_playlists, validation_tracks = model(adjacency)
        validation_ndcg = quick_validation_ndcg(
            validation_playlists.detach().cpu().numpy(), validation_tracks.detach().cpu().numpy()
        )
    elapsed = time.perf_counter() - started
    history.append({
        "epoch": epoch,
        "loss": float(loss.detach().cpu()),
        "ranking_loss": float(ranking_loss.detach().cpu()),
        "validation_ndcg_at_10": validation_ndcg,
        "seconds": elapsed,
    })
    print(f"epoch={epoch:02d} loss={loss.detach().cpu().item():.5f} validation_NDCG@10={validation_ndcg:.5f} time={elapsed:.1f}s")

    if validation_ndcg > best_validation_ndcg + MIN_VALIDATION_IMPROVEMENT:
        best_validation_ndcg = validation_ndcg
        best_epoch = epoch
        best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {epoch} epochs; best epoch was {best_epoch}.")
            break

history_pdf = pd.DataFrame(history)
history_pdf.to_csv(REPORT_ROOT / "lightgcn_training_history.csv", index=False)
assert best_state is not None
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    final_playlist_tensor, final_track_tensor = model(adjacency)
final_playlist_embeddings = unit_rows(final_playlist_tensor.cpu().numpy())
final_track_embeddings = unit_rows(final_track_tensor.cpu().numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history_pdf["epoch"], history_pdf["loss"], marker="o")
axes[0].axvline(best_epoch, linestyle="--", color="gray", label=f"best epoch={best_epoch}")
axes[0].set_title("BPR training objective")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(history_pdf["epoch"], history_pdf["validation_ndcg_at_10"], marker="o", color=sns.color_palette("colorblind")[1])
axes[1].axvline(best_epoch, linestyle="--", color="gray")
axes[1].set_title("Validation NDCG@10 during training")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Sampled NDCG@10")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "lightgcn_training_history.png", bbox_inches="tight")
plt.show()
display(history_pdf.round(5))
print(f"Selected epoch: {best_epoch}; validation NDCG@10={best_validation_ndcg:.5f}")

**Observed.** Early stopping is based only on validation NDCG@10. The checkpoint from the best validation epoch is restored before any final comparison; test labels have not yet been scored or inspected.

## 7. Validation baselines and hybrid weight selection

In [ ]:
def seed_centroids(item_embeddings, users=train_users, items=train_items):
    sums = np.zeros((num_playlists, item_embeddings.shape[1]), dtype=np.float32)
    np.add.at(sums, users, item_embeddings[items])
    return unit_rows(sums)


seed_queries = seed_centroids(final_track_embeddings)
audio_seed_sums = np.zeros((num_playlists, audio_features.shape[1]), dtype=np.float32)
audio_training_mask = audio_available[train_items]
np.add.at(audio_seed_sums, train_users[audio_training_mask], audio_features[train_items[audio_training_mask]])
audio_seed_queries = unit_rows(audio_seed_sums)
playlist_has_audio_seed = np.linalg.norm(audio_seed_sums, axis=1) > 0
del audio_seed_sums


def random_scorer(user, candidates):
    rng = np.random.default_rng(SEED + 30_000 + int(user))
    return rng.random(len(candidates))


def popularity_scorer(user, candidates):
    return np.log1p(train_degree[candidates]) - candidates * 1.0e-12


def transductive_scorer(user, candidates):
    return final_track_embeddings[candidates] @ final_playlist_embeddings[user]


def seed_scorer(user, candidates):
    return final_track_embeddings[candidates] @ seed_queries[user]


def hybrid_scorer(weight):
    def score(user, candidates):
        collaborative = final_track_embeddings[candidates] @ seed_queries[user]
        if weight <= 0 or not playlist_has_audio_seed[user]:
            return collaborative
        available = audio_available[candidates]
        output = collaborative.copy()
        audio_score = audio_features[candidates[available]] @ audio_seed_queries[user]
        output[available] = (1 - weight) * output[available] + weight * audio_score
        return output
    return score


validation_rows = []
for model_name, scorer in [
    ("Random", random_scorer),
    ("Train popularity", popularity_scorer),
    ("LightGCN playlist embedding", transductive_scorer),
    ("LightGCN seed centroid", seed_scorer),
]:
    result = ranking_metrics(validation_candidates, scorer)
    result.insert(0, "model", model_name)
    result.insert(1, "partition", "validation")
    validation_rows.append(result)

CONTENT_WEIGHTS = [0.00, 0.05, 0.10, 0.20, 0.35]
content_weight_results = []
for weight in CONTENT_WEIGHTS:
    result = ranking_metrics(validation_candidates, hybrid_scorer(weight), ks=(10,))
    content_weight_results.append({"content_weight": weight, **result.iloc[0].to_dict()})
content_weight_pdf = pd.DataFrame(content_weight_results)
selected_content_weight = float(
    content_weight_pdf.sort_values(["ndcg", "recall"], ascending=False).iloc[0]["content_weight"]
)
selected_hybrid_validation = ranking_metrics(validation_candidates, hybrid_scorer(selected_content_weight))
selected_hybrid_validation.insert(0, "model", f"Hybrid seed + audio (w={selected_content_weight:.2f})")
selected_hybrid_validation.insert(1, "partition", "validation")
validation_rows.append(selected_hybrid_validation)

validation_metrics_pdf = pd.concat(validation_rows, ignore_index=True)
display(content_weight_pdf.round(5))
display(validation_metrics_pdf.round(5))
print(f"Validation-selected content weight: {selected_content_weight:.2f}")

**Observed.** Random and train-popularity rankings provide essential reference points. The transductive playlist embedding measures the trained graph model, while the seed-centroid version matches the future website scenario where only a list of songs is supplied. The audio blend weight is chosen exclusively from validation NDCG@10.

## 8. One-time sealed test evaluation

In [ ]:
# No test candidates were scored before this cell. The model checkpoint and content weight are now fixed.
test_rows = []
for model_name, scorer in [
    ("Random", random_scorer),
    ("Train popularity", popularity_scorer),
    ("LightGCN playlist embedding", transductive_scorer),
    ("LightGCN seed centroid", seed_scorer),
    (f"Hybrid seed + audio (w={selected_content_weight:.2f})", hybrid_scorer(selected_content_weight)),
]:
    result = ranking_metrics(test_candidates, scorer)
    result.insert(0, "model", model_name)
    result.insert(1, "partition", "test")
    test_rows.append(result)

test_metrics_pdf = pd.concat(test_rows, ignore_index=True)
all_metrics_pdf = pd.concat([validation_metrics_pdf, test_metrics_pdf], ignore_index=True)
all_metrics_pdf.to_csv(REPORT_ROOT / "lightgcn_ranking_metrics.csv", index=False)
display(test_metrics_pdf.round(5))

plot_metrics = all_metrics_pdf[all_metrics_pdf["k"] == 10].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_metrics, x="model", y="ndcg", hue="partition", palette="colorblind", ax=axes[0])
axes[0].set_title("Validation and sealed-test NDCG@10")
axes[0].set_xlabel("")
axes[0].set_ylabel("NDCG@10")
axes[0].tick_params(axis="x", rotation=35)
sns.barplot(data=plot_metrics, x="model", y="recall", hue="partition", palette="colorblind", ax=axes[1])
axes[1].set_title("Validation and sealed-test Recall@10")
axes[1].set_xlabel("")
axes[1].set_ylabel("Recall@10")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "lightgcn_model_comparison.png", bbox_inches="tight")
plt.show()

**Observed.** The test table is the unbiased final estimate under the stated sampled-candidate protocol. Model quality should be judged against both random and popularity baselines, while catalog coverage and tail share reveal whether accuracy comes from repeatedly recommending head tracks.

## 9. Evaluation by playlist-size segment

In [ ]:
playlist_band_by_id = playlist_catalog_pdf.set_index("playlist_node_id")["playlist_size_band"].to_dict()
segment_rows = []
hybrid_test_scorer = hybrid_scorer(selected_content_weight)
for segment in ["small_5_9", "medium_10_24", "large_25_99", "very_large_100_plus"]:
    segment_candidates = {
        user: value for user, value in test_candidates.items() if playlist_band_by_id[user] == segment
    }
    result = ranking_metrics(segment_candidates, hybrid_test_scorer, ks=(10,))
    result.insert(0, "playlist_size_band", segment)
    segment_rows.append(result)
segment_metrics_pdf = pd.concat(segment_rows, ignore_index=True)
segment_metrics_pdf.to_csv(REPORT_ROOT / "lightgcn_test_metrics_by_playlist_size.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=segment_metrics_pdf, x="playlist_size_band", y="ndcg", color=sns.color_palette("colorblind")[2], ax=ax)
ax.set_title("Hybrid test NDCG@10 by playlist size")
ax.set_xlabel("Playlist size segment")
ax.set_ylabel("NDCG@10")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "lightgcn_test_by_playlist_size.png", bbox_inches="tight")
plt.show()
display(segment_metrics_pdf.round(5))

**Observed.** Segment metrics should be read with the diversity and content-coverage differences found in 02c. Large playlists provide more training seeds but also contain more hidden positives, so their Recall and NDCG are not expected to match short-playlist results exactly.

## 10. Match-score calibration and deployable `.pt` bundle

In [ ]:
def collect_balanced_calibration_examples(candidate_sets, scorer):
    scores, labels = [], []
    rng = np.random.default_rng(SEED + 50_000)
    for user, (candidates, positives) in candidate_sets.items():
        positive_set = set(int(value) for value in positives)
        negatives = np.array([value for value in candidates if int(value) not in positive_set], dtype=np.int64)
        chosen_negatives = rng.choice(negatives, size=len(positives), replace=False)
        scores.extend(np.asarray(scorer(user, positives), dtype=np.float32).tolist())
        labels.extend([1.0] * len(positives))
        scores.extend(np.asarray(scorer(user, chosen_negatives), dtype=np.float32).tolist())
        labels.extend([0.0] * len(chosen_negatives))
    return np.asarray(scores, dtype=np.float32), np.asarray(labels, dtype=np.float32)


calibration_scores, calibration_labels = collect_balanced_calibration_examples(
    validation_candidates, hybrid_scorer(selected_content_weight)
)
calibration_score_tensor = torch.from_numpy(calibration_scores)
calibration_label_tensor = torch.from_numpy(calibration_labels)
calibration_scale = torch.tensor(1.0, requires_grad=True)
calibration_bias = torch.tensor(0.0, requires_grad=True)
calibration_optimizer = torch.optim.LBFGS(
    [calibration_scale, calibration_bias], lr=0.5, max_iter=100, line_search_fn="strong_wolfe"
)


def calibration_closure():
    calibration_optimizer.zero_grad()
    logits = calibration_scale * calibration_score_tensor + calibration_bias
    loss = torch_f.binary_cross_entropy_with_logits(logits, calibration_label_tensor)
    loss.backward()
    return loss


calibration_optimizer.step(calibration_closure)
fit_calibration_scale = float(calibration_scale.detach())
fit_calibration_bias = float(calibration_bias.detach())
calibrated_match_scores = 100 * torch.sigmoid(
    calibration_scale.detach() * calibration_score_tensor + calibration_bias.detach()
).numpy()
bundle_path = MODEL_ROOT / "lightgcn_playlist_recommender.pt"


def metric_records(dataframe):
    records = []
    for record in dataframe.to_dict(orient="records"):
        records.append({key: (value.item() if hasattr(value, "item") else value) for key, value in record.items()})
    return records


bundle = {
    "format_version": 1,
    "model_type": "HybridLightGCNPlaylistCompletion",
    "state_dict": best_state,
    "final_track_embeddings": torch.from_numpy(final_track_embeddings).half(),
    "audio_features": torch.from_numpy(audio_features).half(),
    "audio_available": torch.from_numpy(audio_available),
    "fit_calibration_scale": fit_calibration_scale,
    "fit_calibration_bias": fit_calibration_bias,
    "config": {
        "num_playlists": int(num_playlists),
        "num_tracks": int(num_tracks),
        "embedding_dim": EMBEDDING_DIM,
        "layers": LAYERS,
        "best_epoch": int(best_epoch),
        "split_seed": SEED,
        "train_ratio_requested": 0.70,
        "validation_ratio_requested": 0.15,
        "test_ratio_requested": 0.15,
        "selected_content_weight": selected_content_weight,
        "candidate_negatives": TEST_NEGATIVES,
        "fit_percent_definition": "validation-calibrated relative fit score; not a probability of user preference",
    },
    "validation_metrics": metric_records(validation_metrics_pdf),
    "test_metrics": metric_records(test_metrics_pdf),
}
torch.save(bundle, bundle_path)

model_card = {
    "model_type": bundle["model_type"],
    "artifact": str(bundle_path.relative_to(PROJECT_ROOT)),
    "catalog": str(track_catalog_path.relative_to(PROJECT_ROOT)),
    "training_device": device_description,
    "supervision": "implicit positive playlist-track links with BPR negative sampling",
    "leakage_control": "validation/test edges excluded before adjacency and degree construction",
    "split": split_counts_pdf.to_dict(orient="records"),
    "selected_epoch": int(best_epoch),
    "selected_content_weight": selected_content_weight,
    "candidate_protocol": f"all hidden positives plus {TEST_NEGATIVES} uniform unseen negatives per playlist",
    "fit_percent_warning": bundle["config"]["fit_percent_definition"],
    "known_limitations": [
        "No playlist edge order or timestamp is available, so this is set completion rather than next-track prediction.",
        "Audio features cover only 2.714% of graph tracks and are missing not at random.",
        "Sampled-candidate metrics are easier than full-catalog ranking and must be labeled as such.",
        "Tracks absent from the training graph have no learned collaborative neighborhood.",
    ],
}
(MODEL_ROOT / "lightgcn_model_card.json").write_text(json.dumps(model_card, indent=2), encoding="utf-8")
print(f"Saved model bundle : {bundle_path.relative_to(PROJECT_ROOT)} ({bundle_path.stat().st_size / 2**20:.1f} MiB)")
print(f"Saved track catalog: {track_catalog_path.relative_to(PROJECT_ROOT)} ({track_catalog_path.stat().st_size / 2**20:.1f} MiB)")
print(f"Calibration scale/bias: {fit_calibration_scale:.4f} / {fit_calibration_bias:.4f}")
print(f"Balanced validation match-score mean: {calibrated_match_scores.mean():.2f}")
print("Match percentage is a validation-calibrated relative fit score; it is not a probability of user preference.")

**Observed.** The `.pt` file contains the best state dictionary, normalized track embeddings, audio vectors, validation-derived calibration parameters, configuration, and recorded metrics. A compressed catalog supplies Spotify IDs, titles, and artists. This separation is practical for a web backend and avoids serializing a fragile whole Python model object.

## 11. Website-style recommendation demonstration

In [ ]:
from scripts.recommender_inference import PlaylistRecommender

training_audio_seed_counts = np.bincount(
    train_users[audio_available[train_items]], minlength=num_playlists
)
demo_playlist = int(np.argmax(training_audio_seed_counts))
demo_training_items = train_items_by_playlist[demo_playlist]
demo_audio_items = demo_training_items[audio_available[demo_training_items]]
demo_seed_indices = demo_audio_items[:5]
demo_seed_ids = track_catalog_pdf.iloc[demo_seed_indices]["spotify_track_id"].astype(str).tolist()
demo_hidden = set(validation_items_by_playlist[demo_playlist].tolist() + test_items_by_playlist[demo_playlist].tolist())

recommender = PlaylistRecommender(bundle_path, track_catalog_path)
demo_recommendations_pdf, demo_metadata = recommender.recommend(demo_seed_ids, k=10)
demo_recommendations_pdf["is_hidden_track_from_demo_playlist"] = demo_recommendations_pdf["track_node_id"].isin(demo_hidden)
demo_seed_pdf = track_catalog_pdf.iloc[demo_seed_indices][
    ["spotify_track_id", "track_title", "artist_name", "audio_features_available"]
].copy()
demo_seed_pdf.to_csv(REPORT_ROOT / "demo_seed_tracks.csv", index=False)
demo_recommendations_pdf.to_csv(REPORT_ROOT / "demo_recommendations.csv", index=False)

print("DEMO INPUT TRACKS")
display(demo_seed_pdf)
print("DEMO RECOMMENDATIONS")
display(demo_recommendations_pdf)
print("Inference metadata:", demo_metadata)

**Observed.** The demonstration accepts only Spotify track IDs, averages their learned collaborative embeddings, optionally adds audio similarity when matched audio exists, excludes the supplied tracks, and returns ranked suggestions. The `match_percent` field is a validation-calibrated relative fit score and must be labeled that way in the website rather than presented as a liking probability.

## 12. Final model hand-off

In [ ]:
best_test_row = (
    test_metrics_pdf[(test_metrics_pdf["model"] == f"Hybrid seed + audio (w={selected_content_weight:.2f})") & (test_metrics_pdf["k"] == 10)]
    .iloc[0]
)
transductive_test_row = (
    test_metrics_pdf[(test_metrics_pdf["model"] == "LightGCN playlist embedding") & (test_metrics_pdf["k"] == 10)]
    .iloc[0]
)
print("MODEL CREATION OBSERVATIONS")
print("===========================")
print(f"1. The final graph split contains {len(train_items):,} training, {len(validation_items):,} validation, and {len(test_items):,} test edges.")
print("2. Validation and test links were removed before train-degree normalization and LightGCN message passing.")
print(f"3. Validation checkpoint selection chose epoch {best_epoch} using NDCG@10={best_validation_ndcg:.5f}; the test partition was still sealed.")
print(f"4. Sealed-test LightGCN playlist-embedding NDCG@10={transductive_test_row['ndcg']:.5f} and Recall@10={transductive_test_row['recall']:.5f}.")
print(f"5. Website-style hybrid seed-centroid NDCG@10={best_test_row['ndcg']:.5f} and Recall@10={best_test_row['recall']:.5f}.")
print(f"6. Hybrid test catalog coverage@10={best_test_row['catalog_coverage']:.3%}; tail recommendation share={best_test_row['tail_recommendation_share']:.3%}.")
print(f"7. Model bundle saved to {bundle_path.relative_to(PROJECT_ROOT)}.")
print(f"8. Inference catalog saved to {track_catalog_path.relative_to(PROJECT_ROOT)}.")
print("9. The website must call scripts/recommender_inference.py and describe match_percent as a validation-calibrated relative fit score, not a liking probability.")
print("10. Online user feedback is not available here; offline hidden-link metrics measure playlist completion, not guaranteed user satisfaction.")

**Next implementation step.** A web API can load the model once with `load_default_recommender(PROJECT_ROOT)`, accept Spotify track IDs, and return the DataFrame records as JSON. The front end should show the suggested title, artist, and relative match score, plus a tooltip explaining that the score is validation calibrated and is not a user-liking probability.